# Launder Simulation Benchmark Notebook
This notebook demonstrates loading simulator exports, computing benchmark statistics, training a baseline model, and running quick LI/HI simulations.

In [ ]:
from pathlib import Path
import pandas as pd
from aml_sim.detection import RiskModel

DATA_DIR = Path('outputs')
edges = pd.read_csv(DATA_DIR / 'graph_edges.csv')
accounts = pd.read_csv(DATA_DIR / 'accounts.csv')
features = pd.read_csv(DATA_DIR / 'graph_features.csv')
print(f"Loaded {len(edges)} edges and {len(accounts)} accounts")

In [ ]:
# Compute summary statistics
risk_model = RiskModel(0.8, {})
edges['risk_score'] = edges.apply(lambda row: risk_model.score_transaction(row['amount'], bool(row.get('is_money_laundering', False)), row.get('illicit_amount', 0.0), row.get('illicit_fraction', 0.0), bool(row.get('cross_bank', False)), bool(row.get('cross_currency', False)), channel=row.get('channel'), tx_type=row.get('tx_type')), axis=1)
class_balance = edges['is_money_laundering'].value_counts(normalize=True)
pattern_counts = edges['ml_pattern'].value_counts().head(5)
features['total_degree'] = features['in_degree'] + features['out_degree']
top_accounts = features.sort_values('total_degree', ascending=False).head(5)
print('Class balance:\n', class_balance)
print('\nTop laundering patterns:\n', pattern_counts)
print('\nTop degree accounts:\n', top_accounts[['account_id','total_degree','fan_in','fan_out']])
print('
Risk score percentiles:
', edges['risk_score'].describe(percentiles=[0.5,0.9,0.99]))

In [ ]:
# Baseline model using the helper script
import sys
sys.path.append(str(Path('.').resolve()))
from examples.baseline import build_feature_table, temporal_split, _fit_baseline, evaluate_model

merged = build_feature_table(edges, features)
train_df, val_df, test_df = temporal_split(merged, day_column='event_day')
if train_df.empty:
    train_df = merged
if val_df.empty:
    val_df = train_df
feature_cols = [c for c in merged.columns if c.startswith(('sender_', 'receiver_'))]
model = _fit_baseline(train_df[feature_cols], train_df['is_money_laundering'].astype(int))
metrics = evaluate_model(model, {'val': val_df, 'test': test_df}, feature_cols)
metrics

In [ ]:
# Quick LI/HI simulation hooks
from aml_sim.config import load_config
from aml_sim.simulation import Simulation

config_path = Path('config/default.yaml')
for scenario in ['LI', 'HI']:
    cfg = load_config(config_path)
    preset = cfg.intensity_presets.get(scenario, cfg.intensity_presets.get(scenario.upper(), {}))
    cfg.scenario = scenario
    cfg.population_scale = preset.get('population_scale', cfg.population_scale)
    cfg.transaction_scale = preset.get('transaction_scale', cfg.transaction_scale)
    cfg.laundering_intensity = preset.get('laundering_intensity', cfg.laundering_intensity)
    cfg.simulation_days = 1
    sim = Simulation(cfg)
    sim.run()
    print(sim.get_daily_summary(as_json=False))
